# Build network

Build or load one topology-only PyPSA network for the selected source, then show a compact summary and map. Demand is attached later in `01_demand_settings.ipynb` and the interruption analysis notebook.

`NETWORK_SOURCE = "base"` uses reviewed `lines.csv` and `generators.csv`. `NETWORK_SOURCE = "inferred"` derives a topology from OSM roads plus any local GridFinder layer for `OSM_REGION`.


## Settings


In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
import pypsa

from mu_star_energy.network_source import build_network
from mu_star_energy.osm import OSMDownloadRequired, region_slug
from mu_star_energy.paths import processed_energy_dir

PROVIDED_DIR = processed_energy_dir() / "provided"
NETWORK_OUTPUT_DIR = processed_energy_dir() / "networks"

# Allowed values: "base" or "inferred".
NETWORK_SOURCE = "inferred"

# Required for NETWORK_SOURCE = "inferred"; ignored for "base".
OSM_REGION = "Mauritius" # Mauritius options:

# None uses "base" or "inferred-<region>".
OUTPUT_NAME = None

# Inferred runs rebuild by default so stale topology files are not displayed.
OVERWRITE = NETWORK_SOURCE == "inferred"

# Set True only when you want to fetch missing OSM cache files.
ALLOW_DOWNLOAD = True

# OSM road detail for inferred builds: "drive" or "all".
OSM_NETWORK_TYPE = "drive"
MAX_ANCHOR_DISTANCE_M = 1000


## Load or derive network


In [ ]:
if NETWORK_SOURCE not in {"base", "inferred"}:
    raise ValueError(f'NETWORK_SOURCE must be "base" or "inferred", not {NETWORK_SOURCE!r}')
if NETWORK_SOURCE == "inferred" and not OSM_REGION:
    raise ValueError('NETWORK_SOURCE = "inferred" requires OSM_REGION')

if OUTPUT_NAME:
    output_stem = OUTPUT_NAME
elif NETWORK_SOURCE == "inferred":
    output_stem = f"inferred-{region_slug(OSM_REGION)}"
else:
    output_stem = "base"

network_path = NETWORK_OUTPUT_DIR / f"{output_stem}.nc"
metadata_path = NETWORK_OUTPUT_DIR / f"{output_stem}_metadata.json"

if network_path.exists() and not OVERWRITE:
    action = "loaded existing"
else:
    try:
        outputs = build_network(
            NETWORK_SOURCE,
            input_dir=PROVIDED_DIR,
            output_dir=NETWORK_OUTPUT_DIR,
            region=OSM_REGION if NETWORK_SOURCE == "inferred" else None,
            output_name=OUTPUT_NAME,
            overwrite=OVERWRITE,
            allow_download=ALLOW_DOWNLOAD,
            network_type=OSM_NETWORK_TYPE,
            max_anchor_distance_m=MAX_ANCHOR_DISTANCE_M,
        )
    except OSMDownloadRequired as err:
        raise RuntimeError(
            "Missing OSM cache for this inferred build. Set ALLOW_DOWNLOAD = True "
            "and rerun this cell if you want to fetch it."
        ) from err
    network_path = outputs.network
    metadata_path = outputs.metadata
    action = "built"

network = pypsa.Network(network_path)
metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}

summary = pd.Series(
    {
        "action": action,
        "source": NETWORK_SOURCE,
        "region": metadata.get("region", OSM_REGION if NETWORK_SOURCE == "inferred" else ""),
        "network": str(network_path),
        "buses": len(network.buses),
        "lines": len(network.lines),
        "generators": len(network.generators),
        "loads": len(network.loads),
        "OSM road edges": metadata.get("road_edges", ""),
        "GridFinder edges": metadata.get("gridfinder_edges", ""),
        "provisional root": metadata.get("provisional_root", ""),
        "service weights": metadata.get("service_weights", ""),
    },
    name="value",
)
display(summary.to_frame())


## Plot


In [ ]:
if network.buses.empty:
    raise ValueError("Network has no buses to plot")

plots_dir = NETWORK_OUTPUT_DIR / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)
static_png_path = plots_dir / f"{output_stem}_network.png"

line_lons = []
line_lats = []
for line in network.lines.itertuples():
    bus0 = network.buses.loc[line.bus0]
    bus1 = network.buses.loc[line.bus1]
    line_lons.extend([bus0.x, bus1.x, None])
    line_lats.extend([bus0.y, bus1.y, None])

fig, ax = plt.subplots(figsize=(9, 9))
for lon0, lat0, lon1, lat1 in zip(line_lons[0::3], line_lats[0::3], line_lons[1::3], line_lats[1::3]):
    ax.plot(
        [lon0, lon1],
        [lat0, lat1],
        color="#64748b",
        linewidth=0.45 if NETWORK_SOURCE == "inferred" else 1.2,
        alpha=0.7,
        zorder=1,
    )

ax.scatter(
    network.buses.x,
    network.buses.y,
    s=8 if NETWORK_SOURCE == "inferred" else 28,
    color="#f97316",
    edgecolors="black",
    linewidths=0.25,
    zorder=2,
)
ax.set_title(f"{output_stem} network")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2)
fig.savefig(static_png_path, dpi=200, bbox_inches="tight")
print(f"Saved matplotlib PNG: {static_png_path}")
plt.show()

lon_span = float(network.buses.x.max() - network.buses.x.min())
lat_span = float(network.buses.y.max() - network.buses.y.min())
span = max(lon_span, lat_span)
if span > 2:
    zoom = 6
elif span > 0.5:
    zoom = 8
elif span > 0.2:
    zoom = 9
elif span > 0.1:
    zoom = 10
else:
    zoom = 11

bus_hover = [
    f"{bus_id}<br>v_nom: {row.get('v_nom', '')}<br>x: {row.x:.5f}<br>y: {row.y:.5f}"
    for bus_id, row in network.buses.iterrows()
]
interactive = go.Figure()
interactive.add_trace(
    go.Scattermap(
        lon=line_lons,
        lat=line_lats,
        mode="lines",
        name="lines",
        line={"color": "#475569", "width": 1 if NETWORK_SOURCE == "inferred" else 2},
        hoverinfo="skip",
    )
)
interactive.add_trace(
    go.Scattermap(
        lon=network.buses.x,
        lat=network.buses.y,
        mode="markers",
        name="buses",
        marker={"size": 5 if NETWORK_SOURCE == "inferred" else 9, "color": "#f97316"},
        text=bus_hover,
        hoverinfo="text",
    )
)
interactive.update_layout(
    title=f"{output_stem} interactive network map",
    height=720,
    margin={"l": 0, "r": 0, "t": 40, "b": 0},
    map={
        "style": "open-street-map",
        "center": {"lon": float(network.buses.x.mean()), "lat": float(network.buses.y.mean())},
        "zoom": zoom,
    },
    legend={"orientation": "h", "yanchor": "bottom", "y": 0.01, "xanchor": "left", "x": 0.01},
)
interactive.show()
